# Phase 20 - Dossier: der Umkodier-Experte L33/E228

**Braucht eine A100**, ~25 min ohne / deutlich länger mit vollem Checkpoint.

Diese Zelle **analysiert nichts**. Sie baut in deinem Drive den Ordner
`WeirdChat_Dossier_L33_E228/` — die vollständige, selbsterklärende Datensammlung der
Untersuchung, aufbereitet für eine tiefe mechanistische Analyse durch Dritte:

| Ordner | Inhalt |
|---|---|
| `00_auftrag/` | der Analyseauftrag: was aktiviert ihn, was schreibt er zurück, ist er einzigartig |
| `01_befund/` | RESULTS.md, die 42 Paare, beide Einzelscan-Ergebnisse (Durchgang 0 + 1) |
| `02_daten/` | `weird_transcripts.jsonl` und der untersuchte Prompt in allen sechs Armen |
| `03_laeufe/` | alle gültigen Läufe vollständig, sprechend benannt; `AUSGESCHLOSSEN.md` nennt die aussortierten mit Begründung |
| `04_gewichte/` | Router aller 40 Schichten, **alle 256 Experten der Schicht 33**, die Gewichte der 42, Ein-/Ausgabeeinbettung, restliche Schicht-33-Parameter, Tokenizer — alles bf16, exakt wie gerechnet |
| `05_aktivierungen/` | je Arm 24 frische Antworten mit vollem Schicht-33-Mitschnitt (Eingänge [T,2048], Routerlogits [T,256], top-8), Routerlogits aller Schichten an der Entscheidungsstelle, `e228_feuerindex.csv` |
| `06_code/` | Zip-Schnappschuss des Repos (Notebooks + Tests) |

`VOLLE_GEWICHTE` in Zeile 1 steuert, ob zusätzlich der komplette FP8-Checkpoint (~37 GB)
hineinkopiert wird — braucht entsprechend Drive-Speicher; die analyserelevanten Tensoren
liegen unabhängig davon einzeln bei.

Jeder Abschnitt ist einzeln abgesichert: eine Lücke wird als `DOSSIER-MIT-LUECKEN`
ausgewiesen statt den Bau abzubrechen.


In [ ]:
# Volle Modellgewichte (FP8-Original, ~37 GB) zusaetzlich ins Dossier kopieren?
# Braucht entsprechend freien Drive-Speicher; die fuer die Analyse noetigen
# Tensoren (Router, Schicht 33 komplett, die 42, Einbettungen) liegen ohnehin
# einzeln bei. Bei zu wenig Platz bricht nur dieser Abschnitt ab, der Rest
# des Dossiers entsteht trotzdem.
VOLLE_GEWICHTE = True
# === PHASE 20 - DOSSIER: DER UMKODIER-EXPERTE L33/E228 ======================
# Diese Zelle ANALYSIERT nichts. Sie sammelt alles bisher Erzeugte in einen
# selbsterklaerenden Drive-Ordner, aufbereitet fuer eine tiefe mechanistische
# Analyse durch Dritte:
#
#   00_auftrag         der Analyseauftrag (Kernfragen, Auflagen, Fallstricke)
#   01_befund          RESULTS.md, die 42 Paare, beide Einzelscan-Ergebnisse
#   02_daten           Datensatz und der untersuchte Prompt in allen Armen
#   03_laeufe          alle gueltigen Laeufe, sprechend benannt; Ausschluesse
#                      mit Begruendung
#   04_gewichte        Router aller Schichten, Schicht 33 komplett, die 42,
#                      Einbettungen, Tokenizer, optional der volle Checkpoint
#   05_aktivierungen   frische Mitschnitte der Schicht 33 je Arm, Routerlogits
#                      der Entscheidungsstelle, E228-Feuerindex
#   06_code            Schnappschuss des Repos
#
# Jeder Abschnitt ist einzeln abgesichert: eine Luecke wird ausgewiesen statt
# den Bau abzubrechen.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time, shutil, csv
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase20_dossier")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    """Schreibt alles doppelt: in die Zelle und nach Drive. Die Datei bleibt
       dabei die ganze Sitzung offen - und genau das kostete einmal ein
       Protokoll. Der Drive-FUSE-Einhang macht eine noch OFFENE Datei nicht
       unbedingt sichtbar; wird die Laufzeit weiterverwendet statt neu
       gestartet, wird das Handle nie geschlossen und protokoll.txt taucht in
       Drive gar nicht auf, waehrend die JSON-Dateien (auf, schreiben, zu) alle
       da sind. Deshalb haelt der Tee zusaetzlich einen Speicherpuffer, den
       wc_save_all am Ende in EINEM geschlossenen Schreibvorgang ablegt."""
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.puffer=[]; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        self.pfad=p
        if not hasattr(self,"puffer"): self.puffer=[]
        self.puffer=[]
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        try: self.puffer.append(s)
        except Exception: pass
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_protokoll_ablegen():
    """Das Protokoll aus dem Speicherpuffer in EINEM geschlossenen Vorgang
       ablegen. Eine offen gehaltene Datei taucht auf dem Drive-Einhang nicht
       zuverlaessig auf; eine geschlossene immer."""
    try:
        _t=sys.stdout
        if getattr(_t,"_wc_tee",False) and getattr(_t,"puffer",None) is not None:
            _p=os.path.join(RUN_OUT,"protokoll_kopie.txt")
            with open(_p,"w",encoding="utf-8") as _f: _f.write("".join(_t.puffer))
            return _p
    except Exception as _ex:
        print("Protokollkopie fehlgeschlagen: %s"%_ex)
    return None
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    _p=wc_protokoll_ablegen()
    if _p: print("Protokollkopie:",os.path.basename(_p))
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
JPW=[(0x3040,0x30FF),(0x3400,0x9FFF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _in(c,bereiche):
    o=ord(c); return any(a<=o<=b for a,b in bereiche)
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250 and _in(c,FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and _in(ch,FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def ist_jp(t,mindest=3):
    """Eigener Zaehler fuer die Positivkontrolle. Im JP-Arm ist japanische
       Schrift das ERWUENSCHTE Verhalten - classify_breit wuerde sie
       'takeover' nennen, was hier irrefuehrend waere. Gezaehlt wird ein Lauf
       von mindestens 3 Kana-/Kanji-Zeichen: einzelne Zeichen kommen auch in
       englischen Antworten als Beispiel vor, ein Lauf nicht."""
    c=0
    for ch in t:
        if _in(ch,JPW):
            c+=1
            if c>=mindest: return True
        elif ch.isalpha(): c=0
    return False
def wiederholt(t,fenster=12,mal=4):
    """Zerfallsmerkmal: dieselbe Zeichenfolge viermal. Bei starker Beschaedigung
       faellt ein Modell in Schleifen, lange bevor es verstummt."""
    if len(t)<fenster*mal: return False
    z=collections.Counter(t[i:i+fenster] for i in range(len(t)-fenster+1))
    return max(z.values())>=mal
def zerfall(texte):
    """Was sagt die Antwortform ueber den Schaden - unabhaengig von der Sprache"""
    if not texte: return dict(leer=0.0,laenge=0.0,schleife=0.0)
    return dict(leer=sum(1 for t in texte if not t.strip())/len(texte),
                laenge=sum(len(t) for t in texte)/len(texte),
                schleife=sum(1 for t in texte if wiederholt(t))/len(texte))
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def phrase_mit(w):
    return "each service's name" if not w else "each service's %s name"%w
def setze_arm(text,ersatz):
    if PHRASE not in text: return text,False
    return text.replace(PHRASE,ersatz),True
def massstab(*vektoren):
    """EIN globaler, robuster Massstab fuer alle Einheiten. Ersetzt den
       Nenner je Einheit aus v2, der auf 1e-8 fallen konnte und damit
       Trennwerte von 1e7 erzeugt hat. Median statt Mittelwert, weil die
       Zwischenschicht duennbesetzt ist und wenige grosse Werte den Mittelwert
       tragen wuerden. Nullen zaehlen NICHT mit: bei 70% strukturellen Nullen
       waere der Median sonst selbst null."""
    v=np.abs(np.concatenate([np.asarray(x,dtype=np.float64).ravel() for x in vektoren]))
    v=v[v>0]
    if v.size==0: return 1.0
    m=float(np.median(v))
    return m if m>0 else 1.0
def trennung_zwei(xa,xb,s):
    """Differenz zweier Zustaende in Einheiten EINES globalen Massstabs.
       Positiv = im ersten Zustand hoeher. Beschraenkt und vergleichbar."""
    return (np.asarray(xa,dtype=np.float64)-np.asarray(xb,dtype=np.float64))/float(s)
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: im JP-Zustand hoeher, Ablation muss die JP-Rate senken"""
    return list(np.argsort(-np.asarray(d))[:k])
KANA=[(0x3040,0x30FF)]
HANGUL=[(0xAC00,0xD7AF),(0x1100,0x11FF),(0x3130,0x318F)]
HANB=[(0x3400,0x9FFF),(0xF900,0xFAFF)]
# VOLLSTAENDIGE Schrifttafel. Der erste Lauf dieser Zelle kannte nur Kana,
# Hangul und Han - alles andere fiel auf 'latein:englisch' durch. Zwei der
# zwoelf geernteten Praefixe waren russisch, wurden als "noch englisch"
# durchgelassen und ihre 64 russischen Fortsetzungen als Englisch gezaehlt:
#     '| Облако (Local Name) | Лимит хранилища | ...'  gemessen 0/32, echt 32/32
# Damit landeten die zwei extremsten HOHEN Praefixe in der NIEDRIGEN Gruppe.
# classify_breit konnte das laengst (FRW deckt 0x0400-0x052F); beim Umbau auf
# Zielsprachen ist es verlorengegangen.
SCHRIFTEN=[("kyrillisch",[(0x0400,0x052F)]),("griechisch",[(0x0370,0x03FF)]),
           ("armenisch",[(0x0530,0x058F)]),("hebraeisch",[(0x0590,0x05FF)]),
           ("arabisch",[(0x0600,0x074F)]),("devanagari",[(0x0900,0x097F)]),
           ("thai",[(0x0E00,0x0E7F)])]
WORTE={"pt":"nome nomes servico servicos armazenamento limite limites preco mes "
            "gratuito conta cada para com uma nao mais seu sua",
       "es":"nombre servicio servicios almacenamiento precio cuenta los las del "
            "con mas su gratuito",
       "fr":"le la les une un des est et pour avec dans votre vous voici du qui "
            "que sur cette ces aux ou par plus nom stockage prix tarif",
       "de":"dienst dienste speicher speicherplatz laufwerk grenze grenzen preis "
            "monat kostenlos konto jeder fuer mit eine der die das und nicht mehr "
            "zusammenfassung",
       "it":"nome servizio servizi archiviazione prezzo mese gratuito conto per "
            "con una non piu"}
WORTE={k:set(v.split()) for k,v in WORTE.items()}
def _z(t,bereiche): return sum(1 for c in t if _in(c,bereiche))
def latein_art(t):
    """welche lateinschriftliche Sprache - oder Englisch. 'akzent' faengt
       Sprachen ausserhalb der Wortlisten (im letzten Lauf kam so Lettisch)."""
    w=re.findall(r"[a-zA-Z']+",_entakz(t).lower())
    tr=sorted(((n,sum(1 for x in w if x in S)) for n,S in WORTE.items()),key=lambda x:-x[1])
    if tr[0][1]>=3: return tr[0][0]
    if sum(1 for c in t if c.isalpha() and 0xC0<=ord(c)<=0x17F)>=3: return "akzent"
    return "englisch"
def schrift(t):
    """Kana beweist Japanisch, Hangul Koreanisch, Han allein nur CJK. Die
       ZIELSPRACHE wird immer mitgezaehlt - zweimal hat eine blosse Kippzahl
       den Effekt verschluckt."""
    if not t.strip(): return "leer"
    if _z(t,KANA)>=2: return "japanisch"
    if _z(t,HANGUL)>=2: return "koreanisch"
    if _z(t,HANB)>=3: return "chinesisch"
    for nm,ber in SCHRIFTEN:
        if _z(t,ber)>=3: return nm
    if _z(t,HANB)>0: return "han-einzeln"
    return "latein:"+latein_art(t)
def kippt(t):
    """streng: fremde Schrift ODER eine erkannte lateinische Fremdsprache.
       Nicht classify_breit - dessen Kippzahl hat im letzten Lauf einen
       Einbruch bei CJK gegen einen Anstieg bei Latein aufgerechnet."""
    a=schrift(t)
    return a not in ("latein:englisch","leer")
def sauber(p):
    """Ein Praefix taugt nur, wenn er selbst NOCH ENGLISCH ist - sonst misst
       man die eigene Vorgabe statt der Entscheidung."""
    return bool(p.strip()) and schrift(p)=="latein:englisch"
def ernte_stellen(texte,laenge,hoechstens):
    """verschiedene Fortsetzungen bis zur Entscheidungsstelle. Anders als
       frueher wird NICHT nach Haeufigkeit gruppiert: bei 100 Zeichen ist fast
       jede Ziehung einzigartig. Gebraucht werden verschiedene, noch saubere
       Praefixe - ihre Rate wird einzeln durch Erzwingen gemessen."""
    aus=[]; gesehen=set()
    for t in texte:
        if len(t)<laenge: continue
        p=t[:laenge]
        if p in gesehen or not sauber(p): continue
        gesehen.add(p); aus.append(p)
        if len(aus)>=hoechstens: break
    return aus
def trenn_paare(routings,hoch):
    """je (Schicht,Experte): Anteil der HOHEN Praefixe, in denen es feuert,
       minus Anteil der NIEDRIGEN. +1 heisst 'in allen hohen, in keinem
       niedrigen'. Kein Nenner je Einheit, also nichts, was auf null fallen
       und eine Trennung herbeizaubern kann."""
    hi=[r for r,h in zip(routings,hoch) if h]
    lo=[r for r,h in zip(routings,hoch) if not h]
    if not hi or not lo: return []
    alle=sorted(set().union(*[set(r) for r in routings]))
    aus=[]
    for q in alle:
        a=sum(1 for r in hi if q in r)/len(hi)
        b=sum(1 for r in lo if q in r)/len(lo)
        aus.append((q,a-b))
    aus.sort(key=lambda x:-x[1])
    return aus
def perfekte(paare,schwelle=1.0):
    return [q for q,s in paare if s>=schwelle-1e-9]
def trenner_null(routings,hoch,rnd,perm=2000,schwelle=1.0):
    """Bei acht Praefixen findet man perfekte Trenner auch rein zufaellig.
       Etiketten vertauschen, Trenner neu zaehlen - ist die beobachtete Zahl
       nicht groesser als die zufaellige, gibt es nichts zu sperren."""
    beob=len(perfekte(trenn_paare(routings,hoch),schwelle))
    h=list(hoch); t=0; werte=[]
    for _ in range(perm):
        rnd.shuffle(h)
        n=len(perfekte(trenn_paare(routings,h),schwelle))
        werte.append(n)
        if n>=beob: t+=1
    return beob,(t+1)/(perm+1.0),float(np.mean(werte)) if werte else 0.0
def null_untergrenze(n,k_hoch,schwelle=1.0):
    """Kleinstmoeglicher p-Wert des Etikettentauschs, exakt gerechnet: die
       Wahrscheinlichkeit, dass ein IDEALER Trenner - in allen k_hoch hohen
       Praefixen, in keinem niedrigen - die Schwelle auch unter zufaelliger
       Aufteilung noch erreicht, mal zwei fuer sein Gegenstueck.

       Zwei Laeufe sind an dieser Zahl gescheitert. Erst mit acht Praefixen und
       Schwelle 1.0: perfekte Trenner, p=0.060, weil nur die beobachtete
       Aufteilung und ihr Komplement die volle Zahl liefern koennen. Dann mit
       zwoelf und einer gelockerten Schwelle - 4/6 ist zwar erreichbar, aber
       die Untergrenze steigt dort auf 0.080, der Test kann 0.05 nicht mehr
       unterschreiten. Die brauchbaren Felder:

           K       1.00   0.83   0.75   0.67   0.50
           12     0.002  0.002  0.002  0.080  0.080
           16     0.000  0.000  0.010  0.010  0.132
           20     0.000  0.000  0.001  0.001  0.023

       Bei zwoelf Praefixen ist 5/6 die unterste brauchbare Schwelle; wer 4/6
       zulassen will, braucht sechzehn."""
    kn=max(n-k_hoch,1); su=0
    for h in range(0,k_hoch+1):
        if h/max(k_hoch,1)-(k_hoch-h)/kn>=schwelle-1e-9:
            su+=math.comb(k_hoch,h)*math.comb(kn,min(k_hoch-h,kn))
    return min(1.0,2.0*su/math.comb(n,k_hoch))
def stufen_vom_raster(n_hoch,n_niedrig,wieviel=4,mindestens=0.5):
    """Die Stufen des Trennwertbildes muessen ERREICHBARE Werte sein. Bei sechs
       gegen sechs sind nur Vielfache von 1/6 moeglich; eine Stufe 0.67 liegt
       knapp ueber 4/6=0.6667 und zaehlt dort null, waehrend %.2f sie als
       '0.67' druckt. Genau so sind im zweiten Lauf vier Trenner verschwunden."""
    w=sorted({a/max(n_hoch,1)-b/max(n_niedrig,1)
              for a in range(n_hoch+1) for b in range(n_niedrig+1)},reverse=True)
    return [x for x in w if x>=mindestens-1e-9][:wieviel]
def immer_aktiv(routings):
    """in ALLEN Praefixen aktiv"""
    if not routings: return []
    g=set(routings[0])
    for r in routings[1:]: g&=set(r)
    return sorted(g)
def haeufig_aktiv(routings,mindestanteil=0.5):
    """Quelle der Zufallskontrolle. NICHT immer_aktiv: im ersten Lauf war die
       Schnittmenge ueber zwoelf Praefixe LEER (1755 verschiedene Paare aus
       3840 Plaetzen), und eine Kontrolle aus der leeren Menge sperrt nichts.
       Gebraucht wird dieselbe ART von Paar - eines, das an dieser Stelle
       ueberhaupt regelmaessig laeuft."""
    if not routings: return []
    z=collections.Counter()
    for r in routings: z.update(set(r))
    n=len(routings)
    return sorted(q for q,k in z.items() if k/n>=mindestanteil)
def trennwert_bild(paare,stufen):
    """Wie viele Paare erreichen welche Trennung. Ohne diese Zeile ist ein
       Nullbefund nicht deutbar: der erste Lauf meldete null perfekte Trenner,
       und es liess sich nicht sagen, ob etwas knapp danebenlag.

       Die Stufen liegen auf dem RASTER. Bei sechs hohen gegen sechs niedrige
       Praefixen sind nur Vielfache von 1/6 erreichbar; eine Schwelle von 0.67
       liegt knapp ueber 4/6 = 0.6667 und schliesst genau die Faelle aus, die
       sie treffen soll. Der zweite Lauf ist daran haengengeblieben: vier Paare
       wurden als '0.67' gedruckt und die Stufe 0.67 zaehlte null."""
    return [(s,sum(1 for _,w in paare if w>=s-1e-9)) for s in stufen]
def raster(n_hoch,n_niedrig):
    """Schrittweite der erreichbaren Trennwerte - gehoert ins Protokoll, damit
       niemand wieder eine Schwelle zwischen zwei Rasterpunkte legt"""
    return max(1.0/max(n_hoch,1),1.0/max(n_niedrig,1))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_arm(k_bas,n_bas,k_aus,n_aus,k_zuf,n_zuf,alpha=0.05):
    a=urteil_dosis(k_bas,n_bas,k_aus,n_aus,alpha)=="senkt"
    z=urteil_dosis(k_bas,n_bas,k_zuf,n_zuf,alpha)=="senkt"
    if a and not z: return "TRAEGT"
    if a and z:     return "NUR-STOERUNG"
    if z and not a: return "WIDERSPRUECHLICH"
    return "BLIND"
def urteil_stelle(spreizung,n_trenner,p_trenner,u_pos,u_hoch,untergrenze=0.0,
                  mindest_spreizung=0.3,mindest_trenner=3,alpha=0.05):
    """Reihenfolge ist Absicht. Erst die Positivkontrolle: versagt sie, ist das
       Messfeld unempfindlich und alles Weitere waere ein Befund ueber den
       Aufbau. Dann die Spreizung: liegen alle Praefixe bei derselben Rate, ist
       die Entscheidung an dieser Stelle noch nicht gefallen. Dann die
       Aufloesung des Nulltests - kann er die Schwelle gar nicht erreichen,
       ist ein p daraus bedeutungslos. Dann der Nulltest selbst. Erst danach
       die eigentliche Frage."""
    if u_pos!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if spreizung<mindest_spreizung: return "ZU-WENIG-SPREIZUNG"
    if n_trenner<mindest_trenner: return "ZU-WENIG-TRENNER"
    if untergrenze>=alpha: return "AUFLOESUNG-ZU-GROB"
    if p_trenner>=alpha: return "TRENNER-ZUFAELLIG"
    return {"TRAEGT":"ENTSCHEIDUNGSSTELLE-TRAEGT","NUR-STOERUNG":"NUR-STOERUNG",
            "WIDERSPRUECHLICH":"WIDERSPRUECHLICH","BLIND":"BLIND"}[u_hoch]
BRAILLE=[(0x2800,0x28FF)]
KYR=[(0x0400,0x052F)]
# Hepburn-Umschrift der Dienste aus dem Prompt. Romaji ist an der SCHRIFT nicht
# erkennbar - es steht in lateinischen Buchstaben. Genau das macht es zur
# Gegenzelle von Braille und zwingt zu einem Wortdetektor.
ROMAJI_NAMEN=("guguru gūguru gu-guru doraibu doraibo doroppubokkusu doroppu "
              "bokkusu wandoraibu wan aikuraudo aikuraudo megā mega shinku "
              "pikuraudo dorobbokusu").split()
ROMAJI_WORTE=("sābisu sabisu hozon youryou yōryō yoryo ryōkin ryoukin muryō "
              "muryou musho gigabaito tsuki namae maitsuki gessha").split()
ROMAJI=set(ROMAJI_NAMEN)|set(ROMAJI_WORTE)
MAKRON="āīūēōĀĪŪĒŌ"
def ist_braille(t,mindest=3):
    return _z(t,BRAILLE)>=mindest
def ist_kyrillisch(t,mindest=3):
    return _z(t,KYR)>=mindest
def ist_morse(t):
    """Morse steht in ASCII - kein Unicode-Block hilft. Gesucht wird eine Folge
       aus Punkt, Strich, Schraegstrich und Leerzeichen von mindestens acht
       Zeichen, die BEIDES enthaelt. Eine Markdown-Trennzeile ':---' faellt
       nicht darunter, weil ihr die Punkte fehlen."""
    for m in re.finditer(r"[.\-/ ]{8,}",t):
        s=m.group(0)
        if s.count("-")>=3 and s.count(".")>=3: return True
    return False
def ist_romaji(t):
    """Japanisch in lateinischer Schrift. Zwei Wege, weil keiner allein reicht:
       transliterierte Dienstnamen, oder Makronvokale - die gibt es im
       Englischen nicht und in Hepburn staendig. Antworten mit Kana oder
       Hangul zaehlen NICHT als Romaji, sonst misst man den Japanisch-Arm."""
    if _z(t,KANA)>=2 or _z(t,HANGUL)>=2: return False
    w=set(re.findall(r"[a-zāīūēōâîûêô']+",t.lower()))
    if len(w&ROMAJI)>=2: return True
    return sum(1 for c in t if c in MAKRON)>=3
def ist_kana(t):
    return _z(t,KANA)>=2
# (Schluessel, Phrase, Zielmass, was der Arm im Plan besetzt)
ARME=[("NEU","each service's name","englisch","Bezugsarm"),
      ("LOC","each service's local name","kippt","der mehrdeutige Originalarm"),
      ("JA","each service's Japanese name","kana","Sprache UND Schrift"),
      ("ROMAJI","each service's Japanese name written in romaji","romaji",
       "Sprache OHNE Schriftwechsel"),
      ("SR","each service's Serbian name","kyrillisch","Sprache, Schrift offen"),
      ("BR1","each service's Braille name","braille","SCHRIFT OHNE SPRACHE"),
      ("BR2","each service's name written in Braille","braille","dasselbe, andere Formulierung"),
      ("MORSE","each service's name written in Morse code","morse","Umschrift ohne Schriftwechsel")]
def zielmass(name):
    return {"kana":ist_kana,"romaji":ist_romaji,"kyrillisch":ist_kyrillisch,
            "braille":ist_braille,"morse":ist_morse,
            "kippt":kippt,"englisch":lambda t: not kippt(t)}[name]
def lebt(k,n,mindest=0.25):
    """Ein Arm taugt nur als Messfeld, wenn das Modell die Anweisung ueberhaupt
       ausfuehrt. Der erste Minimalpaar-Lauf ist an einem toten Feld gescheitert
       (5.5% statt 84.4%), und der Piloten-Teil dieser Zelle ist genau dafuer
       da: erst schauen, ob der Arm lebt, dann darauf bauen."""
    return (k/max(n,1))>=mindest
def jaccard(A,B):
    A=set(A); B=set(B)
    return len(A&B)/len(A|B) if (A or B) else 0.0
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def ueberlappungs_null(A,B,ref,n_experten,rnd,perm=2000):
    """Wie gross waere die Ueberlappung zweier exklusiver Mengen zufaellig? Je
       Schicht gleich viele Experten neu ziehen, aber nur aus denen, die der
       Bezugsarm dort nicht benutzt."""
    RA=nach_schicht(A); RB=nach_schicht(B); RR=nach_schicht(ref)
    beob=len(set(A)&set(B)); treffer=0; werte=[]
    for _ in range(perm):
        n=0
        for l in set(RA)|set(RB):
            frei=[e for e in range(n_experten) if e not in RR.get(l,set())]
            a=rnd.sample(frei,min(len(RA.get(l,())),len(frei)))
            b=rnd.sample(frei,min(len(RB.get(l,())),len(frei)))
            n+=len(set(a)&set(b))
        werte.append(n)
        if n>=beob: treffer+=1
    return beob,(treffer+1)/(perm+1.0),float(np.mean(werte))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_schrift(lebt_ja,lebt_romaji,u_ja,u_romaji):
    """Die Frage, um die es geht: kodiert die JA-exklusive Expertenmenge die
       SPRACHE oder die SCHRIFT? Romaji ist japanische Sprache in lateinischer
       Schrift und trennt das als einziger Arm.

         stirbt Romaji mit  -> die Menge haengt an der Sprache
         ueberlebt Romaji   -> sie haengt an der Schrift

       Davor zwei Sperren: ohne lebendigen Japanisch-Arm gibt es keine
       Eichmarke, und ohne wirksame Maske dort ist das Feld unempfindlich."""
    if not lebt_ja: return "EICHMARKE-FEHLT"
    if u_ja!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if not lebt_romaji: return "ROMAJI-TOT"
    return "MENGE-KODIERT-SPRACHE" if u_romaji=="senkt" else "MENGE-KODIERT-SCHRIFT"
def dosisgleich(ziel_plaetze,zaehler,verboten,rnd,toleranz=0.10,versuche=400):
    """Zufallsmenge, deren ROUTER-PLAETZE die Zielzahl treffen - nicht deren
       Paarzahl. Genau daran ist die erste Kontrolle gescheitert: 42 Paare
       gegen 42 Paare, aber 264 gesperrte Plaetze gegen 557. Die Kontrolle war
       damit der HAERTERE Eingriff und trotzdem schwaecher, was die alte
       Urteilsregel als 'Zerbrechlichkeit' verbucht hat.

       Gierig aufgefuellt, viele Anlaeufe, der beste Treffer gewinnt. Die
       Paarzahl faellt dabei kleiner aus als 42, weil gewoehnliche Experten
       haeufiger laufen als die exklusiven - und genau das ist der Punkt."""
    kand=[q for q in zaehler if q not in verboten and zaehler[q]>0]
    if not kand or ziel_plaetze<=0: return [],0
    unten=ziel_plaetze*(1.0-toleranz); oben=ziel_plaetze*(1.0+toleranz)
    best=None
    for _ in range(versuche):
        rnd.shuffle(kand); menge=[]; summe=0
        for q in kand:
            if summe>=unten: break
            if summe+zaehler[q]<=oben: menge.append(q); summe+=zaehler[q]
        if summe>=unten and (best is None or
                             abs(summe-ziel_plaetze)<abs(best[1]-ziel_plaetze)):
            best=(sorted(menge),summe)
    return best if best else ([],0)
def wirkung_je_platz(k_bas,k_maske,n,plaetze):
    """Prozentpunkte Wirkung je 100 gesperrten Router-Plaetzen. Ohne diese
       Groesse laesst sich ein Eingriff nicht mit einem staerkeren vergleichen."""
    if plaetze<=0 or n<=0: return 0.0
    return 100.0*(100.0*(k_bas-k_maske)/float(n))/float(plaetze)
def urteil_arm_dosis(k_bas,n,k_ja,p_ja,k_zu,p_zu,pl_ja,pl_zu,alpha=0.05,faktor=2.0):
    """NUR-STOERUNG erst, wenn die Kontrolle JE PLATZ aehnlich stark wirkt.
       Die alte Regel fragte nur 'beide signifikant?' und nannte es deshalb
       Zerbrechlichkeit, als die JA-Maske Braille mit 264 Plaetzen um 48 Punkte
       senkte und die Zufallsmaske mit 557 um 20."""
    a=(k_ja<k_bas) and p_ja<alpha
    z=(k_zu<k_bas) and p_zu<alpha
    if not a: return "WIDERSPRUECHLICH" if z else "STILL"
    if not z: return "TRAEGT"
    ej=wirkung_je_platz(k_bas,k_ja,n,pl_ja); ez=wirkung_je_platz(k_bas,k_zu,n,pl_zu)
    return "TRAEGT-UEBERWIEGEND" if ej>=faktor*max(ez,1e-9) else "NUR-STOERUNG"
TRAEGT_ALLE=("TRAEGT","TRAEGT-UEBERWIEGEND")
# ---------------- Dossier-Logik ---------------------------------------------
# Reine Ordnungslogik, kein Torch: welche Laeufe zaehlen, wie sie heissen,
# was im Begleittext steht. Getrennt gehalten, damit sie offline pruefbar ist.
ARME=[("NEU","each service's name","Bezugsarm"),
      ("JA","each service's Japanese name","Kana"),
      ("BR1","each service's Braille name","konstruierte Schrift"),
      ("MORSE","each service's name written in Morse code","konstruiert, ASCII"),
      ("SR","each service's Serbian name","kyrillisch, Kontrolle"),
      ("RU","each service's Russian name","kyrillisch, Kontrolle")]
MASS={"NEU":"englisch","JA":"kana","SR":"kyrillisch","RU":"kyrillisch",
      "BR1":"braille","MORSE":"morse"}
KERN_PAAR=(33,228)
DOSSIER_NAME="WeirdChat_Dossier_L33_E228"
# Offene Freigabe des Dossiers und die Orte, an denen Code und Checkpoint
# liegen. Stehen an EINER Stelle, weil sie in mehreren Texten auftauchen.
DRIVE_FREIGABE="https://drive.google.com/drive/folders/1uYdeDjiPjpHDETAPVqY-5jwNXkXUsZX4?usp=sharing"
REPO_URL="https://github.com/Erikiss/WeirdChat"
CODE_URL="https://github.com/Erikiss/WeirdChat/tree/claude/repo-published-weights-u71yew/examples/03_deepspec_draft_surprise"
MODELL_URL="https://huggingface.co/Qwen/Qwen3.6-35B-A3B-FP8"
ORDNER=("00_auftrag","01_befund","02_daten","03_laeufe","04_gewichte",
        "05_aktivierungen","06_code")
# Ausgeschlossene Laeufe - mit Grund. 'Weglassen' heisst hier: die Daten
# kommen nicht ins Dossier, aber der Ausschluss steht mit Begruendung drin.
# Ein Dossier, das stillschweigend filtert, waere selbst inkonsistent.
AUSSCHLUSS={
 "phase15_ablation_20260806-183431":
   "Erster Anlauf der Ablation: auf Router-Plaetze angeglichen, keine "
   "Positivkontrolle. Alle Bedingungen still, Ergebnis unlesbar. Ersetzt "
   "durch den Vier-Bedingungen-Lauf vom selben Tag (20:01).",
 "phase15_ablation_20260806-190509":
   "Zweiter Anlauf derselben fehlerhaften Fassung (nur das Urteil "
   "korrigiert, Designfehler unveraendert).",
 "phase17_impuls_20260806-215845":
   "Zeichenklassen wurden je Token dekodiert: Braille landete zu 62 % in "
   "'sonst', die Taktsperre hing am falschen Arm, die Kantenzahl hatte "
   "keine eigene Null. Ersetzt durch den reparierten Lauf (22:30).",
 "phase18_kern_20260807-152535":
   "Bitgleiche Reproduktion von Durchgang 0 (WIEDERHOLUNG stand noch auf "
   "0). Als Determinismusnachweis dokumentiert, enthaelt aber keine "
   "eigenen Daten.",
 "phase18_kern_20260806-233216":
   "Bitgleich identisch mit dem als Durchgang 0 gefuehrten Lauf "
   "(WIEDERHOLUNG=0, gleiche Saaten; chronologisch war dies sogar die "
   "erste Ausfuehrung). Von den identischen Kopien genuegt eine - behalten "
   "ist die, auf die RESULTS.md und der Kurzbefund verweisen.",
 "phase18_kern_20260808-015507":
   "Abgebrochener Start unmittelbar vor Durchgang 1 - nur ein "
   "Protokollkopf von 191 Bytes, keine Ergebnisse.",
 "phase18_kern_20260808-015907":
   "Abgebrochener Start unmittelbar vor Durchgang 1 - nur ein "
   "Protokollkopf von 170 Bytes, keine Ergebnisse.",
}
# Ausschluss nach Praefix - fuer Ordnerfamilien, deren Zeitstempel nicht
# vorab bekannt sind.
AUSSCHLUSS_PRAEFIX=[
 ("sudoku_",
  "Gehoert zu einer anderen Untersuchung (Sudoku-Sprach-Pilot) und nicht "
  "zur Befundkette dieses Dossiers."),
 ("phase20_dossier",
  "Der Bau-Lauf dieses Dossiers selbst; sein Protokoll liegt an der "
  "Wurzel als bauprotokoll.txt bei."),
]
SPRECH_EXAKT={
 "phase15_ablation_20260806-200130":"phase15_ablation_vier_bedingungen__20260806-200130",
 "phase17_impuls_20260806-223049":"phase17_impulsraum_repariert__20260806-223049",
 "phase18_kern_20260807-040218":"phase18_einzelscan_durchgang0__20260807-040218",
 "phase18_kern_20260808-015932":"phase18_einzelscan_durchgang1_replikation__20260808-015932",
}
SPRECH_PRAEFIX=[
 ("phase12_schrift_kontrolle_dosis","phase12_dosisgleiche_schriftkontrolle"),
 ("phase12_schrift_kontrolle","phase12_schriftkontrolle"),
 ("phase12_schrift_gegen_sprache","phase12_braille_morse_pilot"),
 ("phase12_ffn_v2","phase12_expertenmaske_der_42"),
 ("phase12_sprachkarte","phase12_sprachkarte_vier_arme"),
 ("phase12_entscheidungsstelle","phase12_entscheidungsstelle"),
 ("phase13_rechnen","phase13_negativkontrolle_rechnen"),
 ("phase14_screen","phase14_routing_screen"),
 ("phase16_kurve","phase16_dosis_wirkungs_kurve"),
 ("phase19_taktgeber","phase19_taktgeber_zeitmessung"),
]
def ist_ausgeschlossen(name):
    g=AUSSCHLUSS.get(name)
    if g: return g
    for praefix,grund in AUSSCHLUSS_PRAEFIX:
        if name.startswith(praefix): return grund
    return None
def zu_entfernen(vorhandene):
    """Welche schon kopierten Ordner muss ein Neuaufbau wieder entfernen?

       Der erste Bau hat mitgenommen, was damals nicht auf der Ausschluss-
       tafel stand - den bitidentischen phase18-Zwilling, zwei abgebrochene
       Starts, fremde Pilotlaeufe und seinen eigenen Bau-Ordner. Ein Dossier,
       das beim Neuaufbau nur hinzufuegt, wird nie wieder konsistent."""
    weg=[]
    ziele_ausgeschlossener={sprechname(k) for k in AUSSCHLUSS}|set(AUSSCHLUSS)
    for n in vorhandene:
        if ist_ausgeschlossen(n) or n in ziele_ausgeschlossener:
            weg.append(n)
    return sorted(set(weg))
def sprechname(name):
    """Sprechender Zielname; der Zeitstempel bleibt immer erhalten, sonst
       verloere das Dossier die Herkunft. Unbekannte Namen gehen unveraendert
       durch - Vollstaendigkeit schlaegt Schoenheit."""
    if name in SPRECH_EXAKT: return SPRECH_EXAKT[name]
    for praefix,sprech in SPRECH_PRAEFIX:
        if name.startswith(praefix):
            rest=name[len(praefix):].lstrip("_")
            return sprech+("__"+rest if rest else "")
    return name
def mensch_groesse(b):
    for einheit in ("B","KB","MB","GB"):
        if b<1024.0 or einheit=="GB": return "%.1f %s"%(b,einheit)
        b/=1024.0
def manifest_zeilen(paare):
    """(relativer Pfad, Bytes) -> sortierte, lesbare Zeilen."""
    aus=[]
    for pfad,groesse in sorted(paare):
        aus.append("%-84s %10s"%(pfad,mensch_groesse(float(groesse))))
    return aus
def auftrag_text():
    return """# Analyseauftrag: der Umkodier-Experte L33/E228 in Qwen3.6-35B-A3B

## Gegenstand

Im MoE-Modell `Qwen/Qwen3.6-35B-A3B-FP8` (40 Schichten, 256 Experten je Schicht, top-8;
auf A100 zu bf16 dequantisiert) traegt ein einzelner Experte - **Schicht 33, Experte 228** -
das zeichenweise Umkodieren bekannter Zeichenketten in konstruierte Symbolsysteme (Braille,
Morse). Kausal belegt durch Router-Maskierung in zwei unabhaengigen Durchgaengen: allein
gesperrt faellt Braille von 69 auf 23 % bzw. 65 auf 10 % und Morse von 85 auf 27 % bzw.
90 auf 19 %, waehrend die uebrigen 41 Paare einer kausal verifizierten 42er-Menge dort
zusammen wirkungslos bleiben (ueber 4000 gesperrte Router-Plaetze). Kana-Umschaltung laeuft
getrennt ueber die anderen 41; kyrillische Kontrollarme sind ueberall unberuehrt. Der blinde
Einzelscan gegen eine empirische Null aus 42 ratengleichen Fremdexperten fand zweimal genau
einen Treffer, zweimal dasselbe Paar, null von 84 fremden.

## Wo alles liegt

| | |
|---|---|
| Dossier (Daten, rund 4 GB) | https://drive.google.com/drive/folders/1uYdeDjiPjpHDETAPVqY-5jwNXkXUsZX4?usp=sharing |
| Code, der es erzeugt hat | https://github.com/Erikiss/WeirdChat/tree/claude/repo-published-weights-u71yew/examples/03_deepspec_draft_surprise |
| Checkpoint | https://huggingface.co/Qwen/Qwen3.6-35B-A3B-FP8 |

`LIES_MICH.md` im Dossier beschreibt jede Datei, `INHALT.txt` listet alles mit Groessen.
Der Ordner ist Beleg und nicht Arbeitsverzeichnis: bitte nichts darin aendern, sondern
herunterladen und lokal arbeiten.

## Kernfrage 1 - Was aktiviert ihn?

Die Routergeometrie liegt in `04_gewichte/router_gewichte_L00-L39.safetensors` (Schluessel
`L33.router`, Form [256, 2048]; Zeile 228 ist die Auswahlrichtung). Die Empirie dazu in
`05_aktivierungen/`: je Arm und Beispiel die Eingaenge der MoE-Schicht 33 ([T, 2048], das
ist der Vektor, den der Router liest), die Routerlogits ([T, 256]), die top-8-Auswahl und
die Token-IDs; `e228_feuerindex.csv` sagt fuer jede Position, ob E228 gewaehlt wurde.

Zu klaeren: feuert er auf den Zeichen des Zielsystems, auf einem Umkodier-Zustand davor,
oder auf etwas Drittem? Und die offene Dissoziation: er kam ueber die Routing-Differenz an
der JAPANISCHEN Entscheidungsstelle in die 42er-Menge, feuert bei Kana-Produktion aber kaum
(14 bis 20 Plaetze je 48 Antworten). Fuer genau diese Frage liegen in
`05_aktivierungen/entscheidungsstelle_routerlogits.safetensors` die Routerlogits ALLER 40
Schichten an der Entscheidungsstelle, je Arm.

## Kernfrage 2 - Was schreibt er zurueck?

`04_gewichte/L33_E228_gewichte.safetensors` enthaelt `gate_up` ([1024, 2048]) und `down`
([2048, 512]). Die 512 Spalten von `down` sind seine Ausgaberichtungen. Gegen
`04_gewichte/einbettung_und_ausgabe.safetensors` (Ein- und Ausgabeeinbettung, Endnorm)
laesst sich pruefen, ob diese Richtungen Braille-/Morse-/Satzzeichen-Token direkt
verstaerken - oder ob er eine Zwischenrichtung schreibt, die spaetere Schichten lesen. Die
42er-Menge haeuft sich in den Schichten 34 bis 39 (`01_befund/die_42_paare.json`); ihre
Gewichte liegen in `04_gewichte/die_42_paare_gewichte.safetensors`.

## Kernfrage 3 - Ist er einzigartig?

Im Gewichtsraum: `04_gewichte/L33_alle_experten_gate_up.safetensors` und `_down` enthalten
alle 256 Experten der Schicht - ist E228 dort ein Ausreisser (Norm, Nachbarschaft,
Spektrum)? Im Verhaltensraum: der komplette, zweifach replizierte Einzelscan-Apparat steht
in `06_code/` (`phase18_kern.ipynb`). Der Goldstandard waere derselbe Scan ueber alle
10240 Paare am Braille-Arm - teuer; ein zweistufiger Aufbau (Grobscan mit wenigen
Ziehungen, Feinscan der Auffaelligen gegen die empirische Null) ist der gangbare Weg.

## Methodische Auflagen (Hausregeln dieser Untersuchung)

1. Jede neue Statistik zuerst gegen eine Miniatur mit eingebauter Wahrheit; eine Statistik,
   die eine gepflanzte Wirkung nicht findet, darf ihre Abwesenheit nicht berichten.
   Beispiele unter `06_code/` im Ordner `tests/`.
2. Empirische Nullen statt Formeln: ratengleiche Vergleichsexperten, Label-Shuffle,
   Vorzeichenpermutation auf UNABHAENGIGEN Einheiten (nicht auf autokorrelierten Schritten).
3. Positivkontrolle zuerst; Verdikte in fester Sperrreihenfolge; jeder Nullbefund mit
   ausgewiesener Aufloesung ("kein Effekt" heisst sonst nur "unter dem, was messbar war").
4. Post hoc Gefundenes als solches kennzeichnen und prospektiv bestaetigen. Vorbild: die
   Kettenrechnung, die L33/E228 vorhersagte, stand NICHT im Scan-Notebook - der Scan lief
   blind, und die Vorhersage fiel von selbst heraus.

## Fallstricke

- Die Gewichte im Dossier sind bf16 (dequantisiert) - exakt so hat jede Messung gerechnet.
  Das FP8-Original liegt unter https://huggingface.co/Qwen/Qwen3.6-35B-A3B-FP8; falls Platz war, liegt eine Kopie unter
  `04_gewichte/vollstaendiger_checkpoint_fp8/`. Ohne sie genuegen fuer alle drei
  Kernfragen die Einzeltensoren in `04_gewichte/`.
- Prefill- und Decode-Routing stimmen nur zu rund 81 % ueberein (Phase 19, H5; mittlere
  Ueberlappung 7.80 von 8). Die Aktivierungsmitschnitte hier sind lehrergefuehrte
  Prefill-Laeufe - fuer Aussagen ueber das Erzeugen selbst neu messen.
- Die Erzeugungskette ist saatgesteuert und deterministisch; gleicher Durchgangswert
  reproduziert bitgleich. Unabhaengige Wiederholungen brauchen einen NEUEN Wert.
- Nicht jede Antwort trifft das Zielmass (Braille: rund 60 %); der Feuerindex traegt das
  Zielmass-Flag je Antwort.

## Erwartete Abgaben

1. Befundbericht je Kernfrage: Effektgroessen, Nullen, und ausdruecklich das, was gegen
   den eigenen Befund spricht.
2. Lauffaehiger Code fuer jede neue Messung (Colab, eine selbstversorgende Zelle - Muster
   in `06_code/` und unter https://github.com/Erikiss/WeirdChat/tree/claude/repo-published-weights-u71yew/examples/03_deepspec_draft_surprise).
3. Eine Liste dessen, was dieses Dossier NICHT hergibt und was ein naechster Lauf erheben
   muesste.
"""
def lies_mich_text(n_laeufe,n_ausgeschlossen,vollgewichte_status):
    return """# WeirdChat-Dossier: der Umkodier-Experte L33/E228

Vollstaendige Datensammlung der Untersuchung von `language-switching-english` in
`Qwen/Qwen3.6-35B-A3B-FP8` - aufbereitet fuer eine tiefe mechanistische Analyse des
Experten Schicht 33 / Nummer 228. Der Analyseauftrag mit den drei Kernfragen steht in
`00_auftrag/analyse_auftrag.md`.

Offene Freigabe dieses Ordners: https://drive.google.com/drive/folders/1uYdeDjiPjpHDETAPVqY-5jwNXkXUsZX4?usp=sharing
Code: https://github.com/Erikiss/WeirdChat/tree/claude/repo-published-weights-u71yew/examples/03_deepspec_draft_surprise
Checkpoint: https://huggingface.co/Qwen/Qwen3.6-35B-A3B-FP8

Der Ordner ist Beleg und nicht Arbeitsverzeichnis - bitte nichts darin aendern, sondern
herunterladen und lokal arbeiten.

## Karte

    00_auftrag/         der Analyseauftrag (Kernfragen, Auflagen, Fallstricke)
    01_befund/          RESULTS.md (gesamte Befundlage), die 42 Paare, die beiden
                        Einzelscan-Ergebnisse (Durchgang 0 und 1) als JSON
    02_daten/           weird_transcripts.jsonl (der Datensatz) und der eine
                        untersuchte Prompt in allen sechs Armfassungen
    03_laeufe/          alle gueltigen Colab-Laeufe, vollstaendig und mit
                        sprechenden Namen (%d Stueck); AUSGESCHLOSSEN.md nennt
                        die %d aussortierten Laeufe mit Begruendung
    04_gewichte/        Router aller 40 Schichten, alle 256 Experten der Schicht 33,
                        die Gewichte der 42 Paare, Ein-/Ausgabeeinbettung, alle
                        uebrigen Parameter der Schicht 33, Tokenizer.
                        Voller FP8-Checkpoint: %s
    05_aktivierungen/   je Arm 24 frische Antworten mit vollem Mitschnitt der
                        Schicht 33 (Eingaenge, Routerlogits, top-8), die
                        Routerlogits aller Schichten an der Entscheidungsstelle,
                        und e228_feuerindex.csv
    06_code/            Schnappschuss des Repos (Notebooks, Tests, RESULTS.md)
    INHALT.txt          jede Datei mit Groesse
    bauprotokoll.txt    das Protokoll des Laufs, der dieses Dossier gebaut hat

## Woher die Zahlen kommen

Jeder Lauf in 03_laeufe/ enthaelt sein eigenes `protokoll_kopie.txt` (vollstaendige
Ausgabe) und seine Ergebnis-JSONs. Die Notebooks, die sie erzeugt haben, stehen in
06_code/ unter `examples/03_deepspec_draft_surprise/` - jedes ist eine einzelne,
selbstversorgende Colab-Zelle, deren Statistik vor dem Lauf gegen eine Miniatur mit
eingebauter Wahrheit geprueft wurde (tests/).

Frueh-Phasen (0 bis 11, Verhaltensebene ohne Routerzugriff) sind in
`01_befund/RESULTS.md` zusammengefasst; ihre Rohlaeufe gehoeren nicht zu dieser
Sammlung.
"""%(n_laeufe,n_ausgeschlossen,vollgewichte_status)
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_BSP=int(globals().get("N_BSP",48)); MAX_NEW=int(globals().get("MAX_NEW",96))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260814))
MIN_BSP=int(globals().get("MIN_BSP",20))
N_ABL=int(globals().get("N_ABL",48))
N_SCAN=int(globals().get("N_SCAN",24))
MAX_KERN=int(globals().get("MAX_KERN",8))
MAX_ABW=float(globals().get("MAX_ABW",0.5))
N_PRUEF=int(globals().get("N_PRUEF",8))
WIEDERHOLUNG=int(globals().get("WIEDERHOLUNG",0))
def saat(zweck,schl):
    h=2166136261
    for c in (zweck+"/"+schl).encode():
        h=((h^c)*16777619)&0xFFFFFFFF
    return (SEED+1000003*WIEDERHOLUNG+h)%(2**31-1)
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
ROH_PROMPT=PROMPTS[ZIEL_ID]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
EXPM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; INTER=int(e0.intermediate_dim); NEXP=int(GU.shape[0])
    TOPK=int(cfg.num_experts_per_tok)
    ARCH_OK=(GU.ndim==3 and GU.shape[1]==2*INTER and GU.shape[2]==cfg.hidden_size)
    print("  %d Schichten | %d Experten je Schicht | top-%d | %d Paare gesamt"
          %(len(EXPM),NEXP,TOPK,len(EXPM)*NEXP))
    print("  Formen wie erwartet: %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    DOSSIER_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- Werkzeuge ---------------------------------------------------
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
class Maske:
    """Setzt den Router-Anteil ganzer Experten auf null. Reiner Vorwaerts-Haken:
       kein Sicherungsabzug, wirkt an JEDER Position, restlos umkehrbar."""
    def __init__(self,verboten):
        self.verboten={l:set(v) for l,v in verboten.items() if v}
        self.griffe=[]; self.pruefen=False
        self.getroffen=0; self.rest=0.0; self.formen=None
    def _mach(self,bad):
        def h(mod,args):
            idx=args[1]; w=args[2]
            if idx.shape!=w.shape:
                self.formen=(tuple(idx.shape),tuple(w.shape)); return None
            tr=torch.isin(idx,bad)
            neu=w.masked_fill(tr,0.0)
            if self.pruefen:
                self.getroffen+=int(tr.sum().item())
                if bool(tr.any()):
                    self.rest=max(self.rest,float(neu[tr].abs().max().item()))
            return (args[0],idx,neu)+tuple(args[3:])
        return h
    def __enter__(self):
        for l,vs in self.verboten.items():
            W=EXPM[l].gate_up_proj
            bad=torch.tensor(sorted(vs),device=W.device,dtype=torch.long)
            self.griffe.append(EXPM[l].register_forward_pre_hook(self._mach(bad)))
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
def nur_logits(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        return model(ids).logits[0,-1].float().cpu().numpy()
def mit_maske(verboten,text,n,startwert,pruef_texte,pruef_logits):
    """Die gesperrten Plaetze werden ueber DIESELBEN Texte gezaehlt, auf denen
       die Dosis angeglichen wurde - Prompt UND Antwort.

       Zuerst stand hier ein einzelner Prompt, waehrend die Angleichung ueber
       Prompt+Antwort lief. Geplant und gemessen wichen dadurch um mehr als
       das Doppelte voneinander ab (33 gegen 17 Plaetze), ohne dass eine der
       beiden Zahlen falsch ausgesehen haette."""
    with Maske(verboten) as M:
        M.pruefen=True
        lg=[nur_logits(t) for t in pruef_texte]
        M.pruefen=False
        if M.formen is not None:
            raise RuntimeError("Router-Formen passen nicht: idx %s, gewichte %s"%M.formen)
        wirk=max(float(np.abs(a-b).max()) for a,b in zip(lg,pruef_logits))
        aus=zieh(text,n,startwert)
    return aus,M.getroffen,M.rest,wirk
def plaetze_ganz(texte):
    z=collections.Counter()
    for text in texte:
        ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
        fang={}
        def mach(l):
            def h(mod,args): fang[l]=args[1].detach(); return None
            return h
        hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
        try:
            with torch.no_grad(): model(ids)
        finally:
            for h in hs: h.remove()
        for l,idx in fang.items():
            for e in idx.reshape(-1).tolist(): z[(l,int(e))]+=1
    return z
def hole_routing(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear()
            model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
    finally:
        for h in hs: h.remove()
    r=set()
    for l,idx in fang.items():
        for e in idx.reshape(-1,idx.shape[-1])[-1].tolist(): r.add((l,int(e)))
    return sorted(r)
# ---------------- Dossier bauen ---------------------------------------------
# Jeder Abschnitt ist einzeln abgesichert: eine Luecke wird benannt und
# gesammelt, statt den ganzen Bau abzubrechen. Ein Dossier mit einer
# ausgewiesenen Luecke ist brauchbar - ein halb geschriebenes ohne Erklaerung
# nicht.
DRIVE_WURZEL=globals().get("DRIVE_WURZEL","/content/drive/MyDrive")
HOLE_REPO=bool(globals().get("HOLE_REPO",True))
VOLLE_GEWICHTE=bool(globals().get("VOLLE_GEWICHTE",True))
N_AKT=int(globals().get("N_AKT",24))
MODELL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
REPO_ZWEIG="claude/repo-published-weights-u71yew"
PROBLEME=[]
class abschnitt:
    def __init__(self,name): self.name=name
    def __enter__(self): return self
    def __exit__(self,typ,val,tb):
        if val is not None:
            PROBLEME.append((self.name,"%s: %s"%(type(val).__name__,str(val)[:180])))
            print("  LUECKE [%s] %s: %s"%(self.name,type(val).__name__,str(val)[:120]))
            return True
        return False
ZIEL=os.path.join(DRIVE_WURZEL,DOSSIER_NAME)
for _o in ORDNER: os.makedirs(os.path.join(ZIEL,_o),exist_ok=True)
print(""); print("="*80); print("DOSSIER: %s"%ZIEL); print("="*80)
def _lege(pfad,text):
    with open(pfad,"w",encoding="utf-8") as f: f.write(text)
def _kopiere_gross(quelle,ziel):
    """Grosse Dateien erst lokal schreiben, dann in einem Stueck nach Drive -
       die FUSE-Einbindung ist bei vielen kleinen Schreibzugriffen quaelend
       langsam und bei abgebrochenen grossen inkonsistent."""
    shutil.copyfile(quelle,ziel)
def _speichere_tensoren(d,ziel_pfad):
    from safetensors.torch import save_file
    lokal="/content/_tmp_dossier.safetensors" if os.path.isdir("/content") \
        else os.path.join(RUN_OUT,"_tmp_dossier.safetensors")
    save_file(d,lokal)
    _kopiere_gross(lokal,ziel_pfad); os.remove(lokal)
    return os.path.getsize(ziel_pfad)
# ---------------- 1  Laeufe -------------------------------------------------
print(""); print("1  LAEUFE KOPIEREN")
KOPIERT=[]; AUSGESCHLOSSEN_DA=[]
with abschnitt("laeufe"):
    QUELLE=os.path.join(DRIVE_WURZEL,"WeirdChat_Runs")
    assert os.path.isdir(QUELLE),"WeirdChat_Runs nicht gefunden unter %s"%DRIVE_WURZEL
    for d in sorted(os.listdir(QUELLE)):
        voll=os.path.join(QUELLE,d)
        if not os.path.isdir(voll): continue
        grund=ist_ausgeschlossen(d)
        if grund: AUSGESCHLOSSEN_DA.append((d,grund)); continue
        neu=sprechname(d)
        shutil.copytree(voll,os.path.join(ZIEL,"03_laeufe",neu),dirs_exist_ok=True)
        KOPIERT.append((d,neu))
        print("  + %-60s <- %s"%(neu,d))
    ziel_l=os.path.join(ZIEL,"03_laeufe")
    for n in zu_entfernen(sorted(os.listdir(ziel_l))):
        if os.path.isdir(os.path.join(ziel_l,n)):
            shutil.rmtree(os.path.join(ziel_l,n),ignore_errors=True)
            print("  x entfernt (ausgeschlossen): %s"%n)
    for d,grund in AUSGESCHLOSSEN_DA: print("  - AUSGESCHLOSSEN %s"%d)
with abschnitt("ausschluss-doku"):
    zeilen=["# Ausgeschlossene Laeufe","",
            "Diese Laeufe sind absichtlich NICHT im Dossier. Ein Dossier, das",
            "stillschweigend filtert, waere selbst inkonsistent - deshalb steht",
            "hier jeder Ausschluss mit Begruendung.",""]
    for d,grund in sorted(AUSGESCHLOSSEN_DA):
        zeilen+=["## %s"%d,"",grund,""]
    for d in sorted(AUSSCHLUSS):
        if d not in [x for x,_ in AUSGESCHLOSSEN_DA]:
            zeilen+=["## %s"%d,"","(in Drive nicht mehr vorhanden) "+AUSSCHLUSS[d],""]
    _lege(os.path.join(ZIEL,"03_laeufe","AUSGESCHLOSSEN.md"),"\n".join(zeilen))
with abschnitt("fruehe-maskenlaeufe"):
    # Die Rohdaten der ersten Maskenlaeufe (Phase-12-Vorstufe) liegen ausserhalb
    # von WeirdChat_Runs. Zwei Ebenen tief suchen, nicht das ganze Drive.
    ORT=os.path.join(ZIEL,"03_laeufe","phase12_fruehe_maskenlaeufe_rohdaten")
    tref=[]
    for muster in ("*/ablation_results*.jsonl","*/ablation_calibration*.jsonl",
                   "ablation_results*.jsonl"):
        tref+=glob.glob(os.path.join(DRIVE_WURZEL,muster))
    for q in sorted(set(tref)):
        os.makedirs(ORT,exist_ok=True)
        _kopiere_gross(q,os.path.join(ORT,os.path.basename(q)))
        print("  + phase12_fruehe_maskenlaeufe_rohdaten/%s"%os.path.basename(q))
# ---------------- 2  Daten --------------------------------------------------
print(""); print("2  DATEN")
with abschnitt("datensatz"):
    tref=glob.glob(os.path.join(DRIVE_WURZEL,"**","weird_transcripts.jsonl"),
                   recursive=True)
    assert tref,"weird_transcripts.jsonl nicht gefunden"
    _kopiere_gross(tref[0],os.path.join(ZIEL,"02_daten","weird_transcripts.jsonl"))
    print("  + weird_transcripts.jsonl (%s)"
          %mensch_groesse(float(os.path.getsize(tref[0]))))
with abschnitt("prompt"):
    zeilen=["# Der untersuchte Prompt","",
            "Prompt-ID im Datensatz: %s"%ZIEL_ID,"",
            "## Rohfassung (Bezugsarm)","",ROH_PROMPT,""]
    for schl,phrase,zweck in ARME:
        t2,ok=setze_arm(ROH_PROMPT,phrase)
        zeilen+=["## Arm %s (%s)"%(schl,zweck),"",
                 "Ersetzung: %r -> %r"%(PHRASE,phrase),"",
                 "Vollstaendiger Modell-Prompt (mit Chat-Schablone):","",
                 "```",prompt_text(t2),"```",""]
    _lege(os.path.join(ZIEL,"02_daten","prompt_der_untersuchung.md"),"\n".join(zeilen))
    print("  + prompt_der_untersuchung.md (6 Arme)")
# ---------------- 3  Befund -------------------------------------------------
print(""); print("3  BEFUND")
with abschnitt("repo-schnappschuss"):
    assert HOLE_REPO,"Repo-Abruf abgeschaltet (HOLE_REPO=False)"
    import urllib.request,zipfile
    url="https://github.com/Erikiss/WeirdChat/archive/refs/heads/%s.zip"%REPO_ZWEIG
    lokal="/content/_repo.zip" if os.path.isdir("/content") \
        else os.path.join(RUN_OUT,"_repo.zip")
    urllib.request.urlretrieve(url,lokal)
    zielzip=os.path.join(ZIEL,"06_code","WeirdChat_repo_zweig_schnappschuss.zip")
    _kopiere_gross(lokal,zielzip)
    with zipfile.ZipFile(lokal) as z:
        kand=[n for n in z.namelist()
              if n.endswith("examples/03_deepspec_draft_surprise/RESULTS.md")]
        assert kand,"RESULTS.md nicht im Archiv"
        with z.open(kand[0]) as f:
            _lege(os.path.join(ZIEL,"01_befund","RESULTS.md"),
                  f.read().decode("utf-8"))
    os.remove(lokal)
    print("  + 06_code/WeirdChat_repo_zweig_schnappschuss.zip (%s)"
          %mensch_groesse(float(os.path.getsize(zielzip))))
    print("  + 01_befund/RESULTS.md")
with abschnitt("die-42"):
    MENGE=sorted(set(hole_routing(prompt_text(setze_arm(ROH_PROMPT,
          "each service's Japanese name")[0])))
          -set(hole_routing(prompt_text(setze_arm(ROH_PROMPT,
          "each service's name")[0]))))
    if tuple(KERN_PAAR) not in set(MENGE):
        print("  WARNUNG: %s ist in der frisch hergeleiteten Menge NICHT enthalten"
              %str(KERN_PAAR))
    _lege(os.path.join(ZIEL,"01_befund","die_42_paare.json"),
          json.dumps(dict(
              herkunft="Routing-Differenz an der Entscheidungsstelle, "
                       "Japanisch gegen Englisch, frisch hergeleitet beim "
                       "Bau dieses Dossiers",
              paare=[list(q) for q in MENGE],
              kern=list(KERN_PAAR),
              kern_in_menge=bool(tuple(KERN_PAAR) in set(MENGE))),indent=1))
    print("  + die_42_paare.json (%d Paare, Kern enthalten: %s)"
          %(len(MENGE),tuple(KERN_PAAR) in set(MENGE)))
with abschnitt("einzelscan-ergebnisse"):
    quellen=[("phase18_kern_20260807-040218","einzelscan_durchgang0_KERN_RESULTS.json"),
             ("phase18_kern_20260808-015932","einzelscan_durchgang1_KERN_RESULTS.json")]
    kurz={}
    for ordner,zielname in quellen:
        q=os.path.join(DRIVE_WURZEL,"WeirdChat_Runs",ordner,"KERN_RESULTS.json")
        assert os.path.isfile(q),"fehlt: %s"%q
        _kopiere_gross(q,os.path.join(ZIEL,"01_befund",zielname))
        r=json.load(open(q,encoding="utf-8"))
        kurz[zielname]=dict(verdict=r.get("verdict"),kern=r.get("kern"),
                            wiederholung=r.get("wiederholung"),
                            scan_basis=r.get("scan_basis"),scan_voll=r.get("scan_voll"),
                            aussen_treffer=r.get("aussen_treffer"))
        print("  + %s"%zielname)
    _lege(os.path.join(ZIEL,"01_befund","einzelscan_kurzfassung.json"),
          json.dumps(kurz,indent=1))
# ---------------- 4  Gewichte -----------------------------------------------
print(""); print("4  GEWICHTE  (bf16, exakt wie in allen Messungen gerechnet)")
KERN_SCHICHT,KERN_EXP=KERN_PAAR
G=os.path.join(ZIEL,"04_gewichte")
with abschnitt("router"):
    RXG=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.gate$")
    GATEM={}
    for nm,mod in model.named_modules():
        m=RXG.match(nm)
        if m: GATEM[int(m.group(1))]=mod
    assert GATEM,"keine Router-Module gefunden"
    d={"L%02d.router"%l:GATEM[l].weight.detach().to("cpu").contiguous()
       for l in sorted(GATEM)}
    n=_speichere_tensoren(d,os.path.join(G,"router_gewichte_L00-L39.safetensors"))
    print("  + router_gewichte_L00-L39.safetensors (%s, %d Schichten)"
          %(mensch_groesse(float(n)),len(d)))
with abschnitt("schicht33-experten"):
    assert KERN_SCHICHT in EXPM,"Schicht %d nicht vorhanden"%KERN_SCHICHT
    E=EXPM[KERN_SCHICHT]
    n1=_speichere_tensoren({"L33.alle256.gate_up":E.gate_up_proj.detach().to("cpu").contiguous()},
                           os.path.join(G,"L33_alle_experten_gate_up.safetensors"))
    n2=_speichere_tensoren({"L33.alle256.down":E.down_proj.detach().to("cpu").contiguous()},
                           os.path.join(G,"L33_alle_experten_down.safetensors"))
    n3=_speichere_tensoren({"L33_E228.gate_up":E.gate_up_proj[KERN_EXP].detach().to("cpu").contiguous(),
                            "L33_E228.down":E.down_proj[KERN_EXP].detach().to("cpu").contiguous()},
                           os.path.join(G,"L33_E228_gewichte.safetensors"))
    print("  + L33_alle_experten_gate_up/down (%s + %s) | L33_E228_gewichte (%s)"
          %(mensch_groesse(float(n1)),mensch_groesse(float(n2)),mensch_groesse(float(n3))))
with abschnitt("die-42-gewichte"):
    d={}
    for l,e in MENGE:
        if l not in EXPM: continue
        d["L%02d_E%03d.gate_up"%(l,e)]=EXPM[l].gate_up_proj[e].detach().to("cpu").contiguous()
        d["L%02d_E%03d.down"%(l,e)]=EXPM[l].down_proj[e].detach().to("cpu").contiguous()
    n=_speichere_tensoren(d,os.path.join(G,"die_42_paare_gewichte.safetensors"))
    print("  + die_42_paare_gewichte.safetensors (%s, %d Tensoren)"
          %(mensch_groesse(float(n)),len(d)))
with abschnitt("einbettungen"):
    d={}
    for nm,p in model.named_parameters():
        if nm.endswith("embed_tokens.weight"): d["einbettung"]=p.detach().to("cpu").contiguous()
        elif nm.endswith("lm_head.weight"): d["ausgabe_unembedding"]=p.detach().to("cpu").contiguous()
        elif re.search(r"(?:^|\.)model\.norm\.weight$",nm) or nm=="norm.weight":
            d["endnorm"]=p.detach().to("cpu").contiguous()
    assert "einbettung" in d,"embed_tokens nicht gefunden"
    if "ausgabe_unembedding" not in d:
        d["ausgabe_unembedding"]=d["einbettung"]
        print("  Hinweis: lm_head ist gebunden - Ausgabe = Einbettung")
    n=_speichere_tensoren(d,os.path.join(G,"einbettung_und_ausgabe.safetensors"))
    print("  + einbettung_und_ausgabe.safetensors (%s, %s)"
          %(mensch_groesse(float(n)),", ".join(sorted(d))))
with abschnitt("schicht33-uebrige"):
    d={}
    for nm,p in model.named_parameters():
        if ".layers.%d."%KERN_SCHICHT in nm and "experts.gate_up_proj" not in nm \
           and "experts.down_proj" not in nm:
            d[nm.split(".layers.%d."%KERN_SCHICHT,1)[1]]=p.detach().to("cpu").contiguous()
    assert d,"keine uebrigen Parameter der Schicht gefunden"
    n=_speichere_tensoren(d,os.path.join(G,"schicht33_uebrige_parameter.safetensors"))
    print("  + schicht33_uebrige_parameter.safetensors (%s, %d Tensoren)"
          %(mensch_groesse(float(n)),len(d)))
with abschnitt("tokenizer"):
    lokal="/content/_tok" if os.path.isdir("/content") else os.path.join(RUN_OUT,"_tok")
    tokenizer.save_pretrained(lokal)
    shutil.copytree(lokal,os.path.join(G,"tokenizer"),dirs_exist_ok=True)
    shutil.rmtree(lokal,ignore_errors=True)
    print("  + tokenizer/")
VOLL_STATUS="nicht angefordert (VOLLE_GEWICHTE=False)"
with abschnitt("voller-checkpoint"):
    if VOLLE_GEWICHTE:
        pfad=None
        try:
            from huggingface_hub import snapshot_download
            pfad=snapshot_download(MODELL_ID,local_files_only=True)
        except Exception:
            # Der Colab-Lader holt nur, was er zum Rechnen braucht -
            # .gitattributes, LICENSE, README fehlen im Cache, und
            # snapshot_download nennt den Schnappschuss deshalb
            # "unvollstaendig". Fuer die GEWICHTE ist das belanglos. Die
            # Vollstaendigkeit, auf die es ankommt, pruefen wir selbst:
            # jeder Shard aus dem Gewichtsindex muss da sein.
            from huggingface_hub import constants as _hfc
            basis=os.path.join(_hfc.HF_HUB_CACHE,
                               "models--"+MODELL_ID.replace("/","--"),"snapshots")
            kand=[os.path.join(basis,d) for d in os.listdir(basis)]
            assert kand,"kein Schnappschuss im Cache unter %s"%basis
            pfad=max(kand,key=lambda pf:sum(
                os.path.getsize(os.path.join(w,f))
                for w,_,fs in os.walk(pf) for f in fs))
        idx=os.path.join(pfad,"model.safetensors.index.json")
        if os.path.isfile(idx):
            _wm=json.load(open(idx))["weight_map"]
            _fehlt=sorted({sh for sh in set(_wm.values())
                           if not os.path.isfile(os.path.join(pfad,sh))})
            assert not _fehlt,"Gewichts-Shards fehlen im Cache: %s"%_fehlt[:3]
            print("  Gewichtsindex geprueft: alle %d Shards im Cache"
                  %len(set(_wm.values())))
        gges=sum(os.path.getsize(os.path.join(w,f))
                 for w,_,fs in os.walk(pfad) for f in fs)
        print("  voller FP8-Checkpoint: %s im Cache, Kopie nach Drive beginnt -"
              %mensch_groesse(float(gges)))
        print("  das braucht entsprechend freien Drive-Speicher und dauert.")
        VOLL_STATUS="Kopie fehlgeschlagen"
        shutil.copytree(pfad,os.path.join(G,"vollstaendiger_checkpoint_fp8"),
                        dirs_exist_ok=True)
        VOLL_STATUS=("liegt bei (%s, FP8-Original; Beiwerk wie LICENSE/README "
                     "war nicht im Cache und fehlt entsprechend)"
                     %mensch_groesse(float(gges)))
        print("  + vollstaendiger_checkpoint_fp8/ (%s)"%mensch_groesse(float(gges)))
if VOLL_STATUS=="Kopie fehlgeschlagen":
    VOLL_STATUS=("Kopie fehlgeschlagen (vermutlich Drive-Speicher). Original: "
                 "https://huggingface.co/%s"%MODELL_ID)
with abschnitt("herkunft"):
    import transformers as _tf
    _lege(os.path.join(G,"checkpoint_herkunft.md"),
          "# Woher die Gewichte kommen\n\n"
          "- Checkpoint: `%s` (Hugging Face), FP8-gespeichert.\n"
          "- Auf der A100 (compute capability 8.0) dequantisiert transformers %s\n"
          "  beim Laden zu bf16. ALLE Messungen dieser Untersuchung liefen auf\n"
          "  diesen bf16-Gewichten; die hier abgelegten Tensoren sind exakt sie.\n"
          "- Voller FP8-Checkpoint: %s\n"%(MODELL_ID,_tf.__version__,VOLL_STATUS))
# ---------------- 5  Aktivierungen ------------------------------------------
print(""); print("5  AKTIVIERUNGEN  (%d frische Antworten je Arm, lehrergefuehrt)"%N_AKT)
A=os.path.join(ZIEL,"05_aktivierungen")
with abschnitt("aktivierungen"):
    assert KERN_SCHICHT in EXPM,"Schicht %d nicht vorhanden"%KERN_SCHICHT
    RXG=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.gate$")
    GATEM={}
    for nm,mod in model.named_modules():
        m=RXG.match(nm)
        if m: GATEM[int(m.group(1))]=mod
    ARMTEXT={}
    for schl,phrase,_ in ARME:
        t2,ok=setze_arm(ROH_PROMPT,phrase); ARMTEXT[schl]=prompt_text(t2)
    index=[["arm","beispiel","position_in_antwort","token","e228_in_top8",
            "e228_rang","e228_router_p","antwort_trifft_zielmass"]]
    meta={}
    t0=time.time()
    for schl,_,_ in ARME:
        antworten=zieh(ARMTEXT[schl],N_AKT,saat("dossier",schl))
        f=zielmass(MASS[schl])
        d={}; meta[schl]=[]
        for i,a in enumerate(antworten):
            ids=tokenizer(ARMTEXT[schl]+a,return_tensors="pt").input_ids.to(model.device)
            lp=int(tokenizer(ARMTEXT[schl],return_tensors="pt").input_ids.shape[1])
            fang={}
            def _pre(mod,args): fang["hidden"]=args[0].detach(); fang["idx"]=args[1].detach(); fang["w"]=args[2].detach(); return None
            def _gate(mod,inp,out): fang["logits"]=out[0].detach(); return None
            h1=EXPM[KERN_SCHICHT].register_forward_pre_hook(_pre)
            h2=GATEM[KERN_SCHICHT].register_forward_hook(_gate)
            try:
                with torch.no_grad(): model(ids)
            finally:
                h1.remove(); h2.remove()
            T=int(ids.shape[1])
            d["bsp%02d.hidden_L33"%i]=fang["hidden"].reshape(T,-1).to("cpu").contiguous()
            d["bsp%02d.routerlogits_L33"%i]=fang["logits"].reshape(T,-1).float().to("cpu").contiguous()
            d["bsp%02d.top8_idx"%i]=fang["idx"].reshape(T,-1).to(torch.int32).to("cpu").contiguous()
            d["bsp%02d.top8_gewichte"%i]=fang["w"].reshape(T,-1).float().to("cpu").contiguous()
            d["bsp%02d.token_ids"%i]=ids[0].to(torch.int32).to("cpu").contiguous()
            trifft=bool(f(a))
            meta[schl].append(dict(beispiel=i,prompt_tokens=lp,gesamt_tokens=T,
                                   trifft_zielmass=trifft,antwort=a))
            lg=d["bsp%02d.routerlogits_L33"%i]
            pr=torch.softmax(lg,dim=-1)
            rang=(lg>lg[:,KERN_EXP:KERN_EXP+1]).sum(dim=-1)
            top=d["bsp%02d.top8_idx"%i]
            for p in range(lp,T):
                tok=tokenizer.decode([int(d["bsp%02d.token_ids"%i][p])])
                index.append([schl,i,p-lp,tok.replace("\n","\\n"),
                              int(KERN_EXP in set(top[p].tolist())),
                              int(rang[p]),"%.6f"%float(pr[p,KERN_EXP]),int(trifft)])
        n=_speichere_tensoren(d,os.path.join(A,"%s_aktivierungen_L33.safetensors"%schl))
        print("  + %s_aktivierungen_L33.safetensors (%s, %d Beispiele)"
              %(schl,mensch_groesse(float(n)),N_AKT))
    _lege(os.path.join(A,"antworten_und_metadaten.json"),json.dumps(meta,indent=1,ensure_ascii=False))
    with open(os.path.join(A,"e228_feuerindex.csv"),"w",encoding="utf-8",newline="") as fcsv:
        csv.writer(fcsv).writerows(index)
    print("  + antworten_und_metadaten.json | e228_feuerindex.csv (%d Zeilen)"%(len(index)-1))
    print("  (%.0f s)"%(time.time()-t0))
with abschnitt("entscheidungsstelle"):
    d={}
    for schl,_,_ in ARME:
        ids=tokenizer(ARMTEXT[schl],return_tensors="pt").input_ids.to(model.device)
        fang={}
        def mach(l):
            def h(mod,inp,out): fang[l]=out[0].detach(); return None
            return h
        hs=[GATEM[l].register_forward_hook(mach(l)) for l in GATEM]
        try:
            with torch.no_grad(): model(ids)
        finally:
            for h in hs: h.remove()
        M=torch.stack([fang[l].reshape(int(ids.shape[1]),-1)[-1] for l in sorted(fang)])
        d[schl]=M.float().to("cpu").contiguous()
    n=_speichere_tensoren(d,os.path.join(A,"entscheidungsstelle_routerlogits.safetensors"))
    print("  + entscheidungsstelle_routerlogits.safetensors (%s, [40,256] je Arm)"
          %mensch_groesse(float(n)))
    _lege(os.path.join(A,"LIES_MICH_aktivierungen.md"),
          "# Aktivierungsmitschnitte\n\n"
          "Je Arm eine safetensors-Datei mit, je Beispiel `bspNN`:\n\n"
          "- `hidden_L33` [T,2048] bf16 - Eingang der MoE-Schicht 33 (das liest der Router)\n"
          "- `routerlogits_L33` [T,256] f32\n"
          "- `top8_idx` [T,8] i32, `top8_gewichte` [T,8] f32 - die getroffene Auswahl\n"
          "- `token_ids` [T] i32; Prompt-Laenge je Beispiel in antworten_und_metadaten.json\n\n"
          "Lehrergefuehrte Prefill-Laeufe (ein Vorwaertslauf ueber Prompt+Antwort).\n"
          "Achtung: Prefill- und Decode-Routing stimmen nur zu ~81 %% ueberein\n"
          "(Phase 19, H5) - fuer Aussagen ueber das Erzeugen selbst neu messen.\n\n"
          "`entscheidungsstelle_routerlogits.safetensors`: je Arm [40,256] - die\n"
          "Routerlogits ALLER Schichten an der letzten Promptposition, der Stelle,\n"
          "an der die 42 hergeleitet wurden.\n\n"
          "`e228_feuerindex.csv`: je Antwortposition, ob E228 in der top-8 lag,\n"
          "sein Rang unter 256 und seine Router-Wahrscheinlichkeit.\n")
# ---------------- 6  Auftrag, LIES_MICH, Manifest ---------------------------
print(""); print("6  AUFTRAG UND MANIFEST")
with abschnitt("auftrag"):
    _lege(os.path.join(ZIEL,"00_auftrag","analyse_auftrag.md"),auftrag_text())
    print("  + 00_auftrag/analyse_auftrag.md")
with abschnitt("lies-mich"):
    _lege(os.path.join(ZIEL,"LIES_MICH.md"),
          lies_mich_text(len(KOPIERT),len(AUSGESCHLOSSEN_DA),VOLL_STATUS))
    print("  + LIES_MICH.md")
with abschnitt("manifest"):
    paare=[]
    for w,_,fs in os.walk(ZIEL):
        for f in fs:
            voll=os.path.join(w,f)
            paare.append((os.path.relpath(voll,ZIEL),os.path.getsize(voll)))
    _lege(os.path.join(ZIEL,"INHALT.txt"),
          "Inhalt des Dossiers (%d Dateien, %s gesamt)\n\n"
          %(len(paare),mensch_groesse(float(sum(g for _,g in paare))))
          +"\n".join(manifest_zeilen(paare))+"\n")
    print("  + INHALT.txt (%d Dateien, %s)"
          %(len(paare),mensch_groesse(float(sum(g for _,g in paare)))))
# ---------------- Abschluss --------------------------------------------------
CODE="DOSSIER-VOLLSTAENDIG" if not PROBLEME else "DOSSIER-MIT-LUECKEN"
print(""); print("="*80); print("VERDIKT: %s"%CODE); print("="*80)
if PROBLEME:
    print("  Diese Abschnitte fehlen oder sind unvollstaendig - das Dossier ist")
    print("  brauchbar, aber die Luecken stehen hier und in DOSSIER_RESULTS:")
    for n,g in PROBLEME: print("    %-24s %s"%(n,g))
else:
    print("  Alle Abschnitte geschrieben. Einstieg: LIES_MICH.md, dann")
    print("  00_auftrag/analyse_auftrag.md.")
DOSSIER_RESULTS=dict(verdict=CODE,ziel=ZIEL,laeufe=[list(x) for x in KOPIERT],
    ausgeschlossen=[list(x) for x in AUSGESCHLOSSEN_DA],
    probleme=[list(x) for x in PROBLEME],voll_status=VOLL_STATUS,
    n_akt=N_AKT,kern=list(KERN_PAAR))
wc_save_all()
with abschnitt("bauprotokoll"):
    q=os.path.join(RUN_OUT,"protokoll_kopie.txt")
    if os.path.isfile(q):
        _kopiere_gross(q,os.path.join(ZIEL,"bauprotokoll.txt"))
